# AfriQBench Notebook 01 — Exact TFIM Reference

This notebook establishes the classical reference layer for the first AfriQBench benchmark. It constructs the transverse-field Ising model (TFIM), computes its exact ground state for small systems, evaluates physically meaningful observables, and generates a reproducible parameter sweep.

The Hamiltonian is

$$H=-J\sum_i Z_i Z_{i+1}-h\sum_i X_i.$$

The initial benchmark uses an open chain and exact diagonalization, which is appropriate for the small problem sizes used as validation references.

## 1. Environment

The following cell makes the local `src/` package visible whether the notebook is launched from the repository root or from the `notebooks/` directory.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from afriqbench.models.tfim import tfim_hamiltonian
from afriqbench.reference.exact import ground_state
from afriqbench.observables import (
    transverse_magnetization,
    nearest_neighbor_zz,
)


## 2. Single reference instance

In [ ]:
n_qubits = 4
J = 1.0
h = 1.0

H = tfim_hamiltonian(n_qubits=n_qubits, J=J, h=h, periodic=False)
energy, state = ground_state(H)

mx = transverse_magnetization(state, n_qubits)
zz = nearest_neighbor_zz(state, n_qubits, periodic=False)

print(f'Ground-state energy: {energy:.10f}')
print(f'Energy per site:      {energy / n_qubits:.10f}')
print(f'Transverse <X>:       {mx:.10f}')
print(f'Mean nearest <ZZ>:    {zz:.10f}')


For `N=4`, `J=1`, `h=1`, open boundaries, the regression reference is approximately:

- $E_0=-4.7587704831$
- $\langle X\rangle=0.8100954855$
- mean $\langle Z_iZ_{i+1}\rangle=0.5061295137$.

## 3. Sweep the field ratio $h/J$

In [ ]:
field_values = np.linspace(0.0, 2.0, 41)
rows = []

for field in field_values:
    H = tfim_hamiltonian(
        n_qubits=n_qubits,
        J=J,
        h=float(field),
        periodic=False,
    )
    energy, state = ground_state(H)

    rows.append({
        'n_qubits': n_qubits,
        'J': J,
        'h': float(field),
        'h_over_J': float(field / J),
        'ground_energy': energy,
        'energy_per_site': energy / n_qubits,
        'transverse_magnetization': transverse_magnetization(state, n_qubits),
        'nearest_neighbor_zz': nearest_neighbor_zz(state, n_qubits),
    })

df = pd.DataFrame(rows)
df.head()


## 4. Ground-state energy

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(df['h_over_J'], df['energy_per_site'], marker='o', markersize=3)
ax.set_xlabel(r'$h/J$')
ax.set_ylabel(r'Ground-state energy per site $E_0/N$')
ax.set_title('Exact TFIM reference: energy, N=4')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## 5. Transverse magnetization

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(df['h_over_J'], df['transverse_magnetization'], marker='o', markersize=3)
ax.set_xlabel(r'$h/J$')
ax.set_ylabel(r'Transverse magnetization $\langle X\rangle$')
ax.set_title('Exact TFIM reference: transverse magnetization, N=4')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## 6. Nearest-neighbour ZZ correlation

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(df['h_over_J'], df['nearest_neighbor_zz'], marker='o', markersize=3)
ax.set_xlabel(r'$h/J$')
ax.set_ylabel(r'Mean $\langle Z_i Z_{i+1}\rangle$')
ax.set_title('Exact TFIM reference: nearest-neighbour correlation, N=4')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## 7. Save the reproducible reference dataset

The generated data can be compared directly with `results/tfim_n4_reference.csv`.

In [ ]:
results_dir = repo_root / 'results'
results_dir.mkdir(exist_ok=True)

output_path = results_dir / 'tfim_n4_reference.csv'
df.to_csv(output_path, index=False, float_format='%.10f')
print(output_path)


## Interpretation

For this finite four-site open chain, increasing the transverse field increases alignment in the X direction while reducing nearest-neighbour Z-order. These are finite-size benchmark trends. AfriQBench does **not** use this small instance by itself to infer the thermodynamic-limit critical point.

The purpose of this notebook is to create trusted classical reference values that later simulator and quantum-hardware runs can be evaluated against using the same benchmark definition.